[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Exceptions as Classes &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.


**1.** A family of three, caught through its base.


In [1]:
class InstrumentError(Exception):
    """Anything that goes wrong with an instrument."""


class CalibrationError(InstrumentError):
    """The instrument reads, but reads wrongly."""


class DisconnectedError(InstrumentError):
    """The instrument is not answering at all."""


for error in [CalibrationError("offset of 3.2 degrees"), DisconnectedError("no reply from sensor 4")]:
    try:
        raise error
    except InstrumentError as caught:
        print(f"caught {type(caught).__name__}: {caught}")


caught CalibrationError: offset of 3.2 degrees
caught DisconnectedError: no reply from sensor 4


One `except InstrumentError` caught both, because each is a kind of `InstrumentError`. A caller who
cares only about calibration can still catch `CalibrationError` on its own.


**2.** An exception that carries its facts.


In [2]:
class CalibrationError(InstrumentError):
    """The instrument reads, but reads wrongly."""

    def __init__(self, instrument, offset):
        super().__init__(f"{instrument} is off by {offset} degrees")
        self.instrument = instrument
        self.offset = offset


try:
    raise CalibrationError("thermometer 2", 3.2)
except CalibrationError as error:
    print("message:   ", error)
    print("instrument:", error.instrument)
    print("offset:    ", error.offset)


message:    thermometer 2 is off by 3.2 degrees
instrument: thermometer 2
offset:     3.2


`super().__init__` hands the message to `Exception`, which is what `str(error)` prints. The two
attributes are there for a handler that needs to act on the facts, such as correcting readings by the
offset, without taking the message apart.


**3.** A second parent, so existing handlers still work.


In [3]:
class DisconnectedError(InstrumentError, ConnectionError):
    """The instrument is not answering at all."""


try:
    raise DisconnectedError("no reply from sensor 4")
except ConnectionError as error:
    print("an existing except ConnectionError caught:", type(error).__name__)

print([c.__name__ for c in DisconnectedError.__mro__])


an existing except ConnectionError caught: DisconnectedError
['DisconnectedError', 'InstrumentError', 'ConnectionError', 'OSError', 'Exception', 'BaseException', 'object']


Code written before `DisconnectedError` existed, catching `ConnectionError`, still catches it. Code
written against the new family catches it through `InstrumentError`.


**4.** Turning a low-level error into one of your own.


In [4]:
class BadValueError(InstrumentError):
    """The instrument sent something that is not a number."""

    def __init__(self, text):
        super().__init__(f"{text!r} is not a number")
        self.text = text


def read_value(text):
    try:
        return float(text)
    except ValueError as error:
        raise BadValueError(text) from error


try:
    read_value("warm")
except BadValueError as error:
    print("raised:   ", error)
    print("__cause__:", type(error.__cause__).__name__, "-", error.__cause__)


raised:    'warm' is not a number
__cause__: ValueError - could not convert string to float: 'warm'


The caller gets an error in the instrument's terms, and `__cause__` keeps Python's original message in
case somebody needs to see exactly what `float` objected to.


**5.** Clause order, wrong and then right.


In [5]:
def handle(error):
    try:
        raise error
    except InstrumentError:
        return "general handler"
    except CalibrationError:
        return "calibration handler"


def handle_fixed(error):
    try:
        raise error
    except CalibrationError:
        return "calibration handler"
    except InstrumentError:
        return "general handler"


print("wrong order:", handle(CalibrationError("thermometer 2", 3.2)))
print("right order:", handle_fixed(CalibrationError("thermometer 2", 3.2)))


wrong order: general handler
right order: calibration handler


In the wrong order the calibration handler never runs, because a `CalibrationError` is an
`InstrumentError` and the first clause matches. Put the most specific class first.


**6.** Skip one kind, stop on another.


In [6]:
def check(value):
    if value is None:
        raise DisconnectedError("a reading is missing")
    if value > 100:
        raise CalibrationError("sensor", value - 100)
    return value


def total(readings):
    kept, skipped = [], 0
    for value in readings:
        try:
            kept.append(check(value))
        except CalibrationError:
            skipped += 1
    return sum(kept), skipped


print("skipped through:", total([12.0, 140.0, 8.5]))

try:
    total([12.0, None, 8.5])
except DisconnectedError as error:
    print("stopped:", error)


skipped through: (20.5, 1)
stopped: a reading is missing


`total` catches only `CalibrationError`, so a reading over 100 is skipped and counted, and a missing
reading raises `DisconnectedError` out of the function, where the caller decides what to do. Catching
`InstrumentError` inside `total` would have skipped the missing reading too.


---

&#8592; **Back to:** [Exceptions as Classes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/13-exceptions-as-classes.ipynb)  &nbsp;&middot;&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
